# Module 7: EvalOps & AI Telemetry for Continuous Quality

Build an evaluation gate that blocks deployment when model quality regresses.


## 1. Setup

Load EvalOps core components.


In [1]:
import sys
from pathlib import Path

module_dir = Path.cwd()
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

from evalops_core import make_golden_dataset, SnapAuditPolicyModel, TelemetryCollector, evaluate_model

print("✓ Setup complete")


✓ Setup complete


## 2. Golden Dataset

Create a deterministic set of 100 verified receipt cases.


In [2]:
dataset = make_golden_dataset(size=100, seed=42)
print("Dataset size:", len(dataset))
dataset[0].to_dict()


2026-03-01 11:56:00 - evalops_core - INFO - Creating golden dataset with 100 cases (seed: 42)
2026-03-01 11:56:00 - evalops_core - INFO - Initialized SnapAuditPolicyModel with variant baseline
2026-03-01 11:56:00 - evalops_core - INFO - Predicting approval for case C001: Coffee amount=$11.9
2026-03-01 11:56:00 - evalops_core - INFO - Case C001 approved: policy compliant
2026-03-01 11:56:00 - evalops_core - INFO - Initialized SnapAuditPolicyModel with variant baseline
2026-03-01 11:56:00 - evalops_core - INFO - Predicting approval for case C002: Coffee amount=$4.12
2026-03-01 11:56:00 - evalops_core - INFO - Case C002 approved: policy compliant
2026-03-01 11:56:00 - evalops_core - INFO - Initialized SnapAuditPolicyModel with variant baseline
2026-03-01 11:56:00 - evalops_core - INFO - Predicting approval for case C003: Travel amount=$160.5
2026-03-01 11:56:00 - evalops_core - INFO - Case C003 approved: policy compliant
2026-03-01 11:56:00 - evalops_core - INFO - Initialized SnapAuditPol

Dataset size: 100


{'case_id': 'C001',
 'category': 'Coffee',
 'total': '11.9',
 'has_receipt': True,
 'suspicious_vendor': False,
 'employee_role': 'Employee',
 'ground_truth_approved': True}

## 3. Baseline Evaluation

Baseline model should meet the 99% threshold.


In [3]:
baseline_model = SnapAuditPolicyModel(variant="baseline")
baseline_telemetry = TelemetryCollector()
baseline_metrics = evaluate_model(dataset, baseline_model, baseline_telemetry)
baseline_metrics


2026-03-01 11:56:01 - evalops_core - INFO - Initialized SnapAuditPolicyModel with variant baseline
2026-03-01 11:56:01 - evalops_core - INFO - Initializing TelemetryCollector
2026-03-01 11:56:01 - evalops_core - INFO - Starting model evaluation for 100 cases with model variant: baseline
2026-03-01 11:56:01 - evalops_core - INFO - Predicting approval for case C001: Coffee amount=$11.9
2026-03-01 11:56:01 - evalops_core - INFO - Case C001 approved: policy compliant
2026-03-01 11:56:01 - evalops_core - INFO - Predicting approval for case C002: Coffee amount=$4.12
2026-03-01 11:56:01 - evalops_core - INFO - Case C002 approved: policy compliant
2026-03-01 11:56:01 - evalops_core - INFO - Predicting approval for case C003: Travel amount=$160.5
2026-03-01 11:56:01 - evalops_core - INFO - Case C003 approved: policy compliant
2026-03-01 11:56:01 - evalops_core - INFO - Predicting approval for case C004: Client Dinner amount=$217.97
2026-03-01 11:56:01 - evalops_core - WARNING - Case C004 denied

{'total_cases': 100,
 'correct': 100,
 'accuracy': 1.0,
 'faithfulness': 1.0,
 'answer_relevance': 1.0,
 'failures': []}

## 4. Regression Simulation

"Friendly" prompt regression that over-approves risky receipts.


In [4]:
regressed_model = SnapAuditPolicyModel(variant="friendly_regression")
regressed_telemetry = TelemetryCollector()
regressed_metrics = evaluate_model(dataset, regressed_model, regressed_telemetry)
regressed_metrics


2026-03-01 11:56:02 - evalops_core - INFO - Initialized SnapAuditPolicyModel with variant friendly_regression
2026-03-01 11:56:02 - evalops_core - INFO - Initializing TelemetryCollector
2026-03-01 11:56:02 - evalops_core - INFO - Starting model evaluation for 100 cases with model variant: friendly_regression
2026-03-01 11:56:02 - evalops_core - INFO - Predicting approval for case C001: Coffee amount=$11.9
2026-03-01 11:56:02 - evalops_core - INFO - Case C001 approved: policy compliant
2026-03-01 11:56:02 - evalops_core - INFO - Predicting approval for case C002: Coffee amount=$4.12
2026-03-01 11:56:02 - evalops_core - INFO - Case C002 approved: policy compliant
2026-03-01 11:56:02 - evalops_core - INFO - Predicting approval for case C003: Travel amount=$160.5
2026-03-01 11:56:02 - evalops_core - INFO - Case C003 approved: policy compliant
2026-03-01 11:56:02 - evalops_core - INFO - Predicting approval for case C004: Client Dinner amount=$217.97
2026-03-01 11:56:02 - evalops_core - INFO

{'total_cases': 100,
 'correct': 84,
 'accuracy': 0.84,
 'faithfulness': 0.84,
 'answer_relevance': 1.0,
 'failures': [{'case': {'case_id': 'C004',
    'category': 'Client Dinner',
    'total': '217.97',
    'has_receipt': True,
    'suspicious_vendor': False,
    'employee_role': 'Employee',
    'ground_truth_approved': False},
   'predicted_approved': True,
   'reason': 'Approved: friendly flexibility for client relationship spend',
   'ground_truth_approved': False},
  {'case': {'case_id': 'C006',
    'category': 'Meal',
    'total': '69.87',
    'has_receipt': True,
    'suspicious_vendor': False,
    'employee_role': 'Manager',
    'ground_truth_approved': False},
   'predicted_approved': True,
   'reason': 'Approved: friendly flexibility for meal claims',
   'ground_truth_approved': False},
  {'case': {'case_id': 'C007',
    'category': 'Client Dinner',
    'total': '183.55',
    'has_receipt': True,
    'suspicious_vendor': False,
    'employee_role': 'Employee',
    'ground_tru

## 5. Compare and Diagnose

Inspect failed cases and telemetry traces.


In [5]:
print("Baseline accuracy:", baseline_metrics["accuracy"])
print("Regressed accuracy:", regressed_metrics["accuracy"])
print("Regression failures:", len(regressed_metrics["failures"]))

regressed_metrics["failures"][:3]


Baseline accuracy: 1.0
Regressed accuracy: 0.84
Regression failures: 16


[{'case': {'case_id': 'C004',
   'category': 'Client Dinner',
   'total': '217.97',
   'has_receipt': True,
   'suspicious_vendor': False,
   'employee_role': 'Employee',
   'ground_truth_approved': False},
  'predicted_approved': True,
  'reason': 'Approved: friendly flexibility for client relationship spend',
  'ground_truth_approved': False},
 {'case': {'case_id': 'C006',
   'category': 'Meal',
   'total': '69.87',
   'has_receipt': True,
   'suspicious_vendor': False,
   'employee_role': 'Manager',
   'ground_truth_approved': False},
  'predicted_approved': True,
  'reason': 'Approved: friendly flexibility for meal claims',
  'ground_truth_approved': False},
 {'case': {'case_id': 'C007',
   'category': 'Client Dinner',
   'total': '183.55',
   'has_receipt': True,
   'suspicious_vendor': False,
   'employee_role': 'Employee',
   'ground_truth_approved': False},
  'predicted_approved': True,
  'reason': 'Approved: friendly flexibility for client relationship spend',
  'ground_truth_

In [6]:
# Recent telemetry events from regressed run
regressed_telemetry.events[-8:]


[{'time': '2026-03-01T16:56:02Z',
  'case_id': 'C097',
  'step': 'start',
  'detail': 'Evaluating Meal amount=18.84'},
 {'time': '2026-03-01T16:56:02Z',
  'case_id': 'C097',
  'step': 'result',
  'detail': 'correct prediction: True'},
 {'time': '2026-03-01T16:56:02Z',
  'case_id': 'C098',
  'step': 'start',
  'detail': 'Evaluating Travel amount=155.84'},
 {'time': '2026-03-01T16:56:02Z',
  'case_id': 'C098',
  'step': 'result',
  'detail': 'correct prediction: True'},
 {'time': '2026-03-01T16:56:02Z',
  'case_id': 'C099',
  'step': 'start',
  'detail': 'Evaluating Travel amount=489.59'},
 {'time': '2026-03-01T16:56:02Z',
  'case_id': 'C099',
  'step': 'result',
  'detail': 'correct prediction: True'},
 {'time': '2026-03-01T16:56:02Z',
  'case_id': 'C100',
  'step': 'start',
  'detail': 'Evaluating Travel amount=830.42'},
 {'time': '2026-03-01T16:56:02Z',
  'case_id': 'C100',
  'step': 'result',
  'detail': 'correct prediction: True'}]

## 6. CI Gate Logic

Block deployment if accuracy drops below 99%.


In [7]:
THRESHOLD = 0.99

def gate(metrics, threshold=THRESHOLD):
    return metrics["accuracy"] >= threshold

print("Baseline gate:", gate(baseline_metrics))
print("Regressed gate:", gate(regressed_metrics))


Baseline gate: True
Regressed gate: False


## 7. Save Artifacts

Persist results and telemetry for observability.


In [8]:
import json

Path("eval_results_baseline.json").write_text(json.dumps(baseline_metrics, indent=2))
Path("eval_results_regressed.json").write_text(json.dumps(regressed_metrics, indent=2))
Path("telemetry_regressed.json").write_text(json.dumps(regressed_telemetry.events, indent=2))

print("✓ Saved eval and telemetry artifacts")


✓ Saved eval and telemetry artifacts
